# RVC v2 48k 训练 - Kaggle

设置 Accelerator 为任一 GPU（推荐 T4 x2），Internet 设为 ON，然后运行全部 Cell。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '未检测到 CUDA GPU'
print(f'[RVC] 检测到 {torch.cuda.device_count()} 张 GPU')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'[RVC] cuda:{i} {props.name} ({props.total_memory / 2**30:.1f} GB)')
print('[RVC] 双卡将自动使用 DDP；单卡将自动使用 cuda:0')

In [ ]:
import subprocess
root = '/kaggle/working/HRVC'
subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/lingrana/rvc_train_kaggle.git', root], check=True)
%cd /kaggle/working/HRVC

In [ ]:
import os, re, subprocess, sys
os.environ['PYTHONNOUSERSITE'] = '1'

CONFLICT_PATTERNS = (
    re.compile(r'^ERROR: pip.s dependency resolver'),
    re.compile(r' requires .+ which is (incompatible|not installed)\.$'),
)

def pip_install_quiet(*packages: str) -> None:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', *packages, '-q'], capture_output=True, text=True)
    if result.returncode != 0:
        sys.stdout.write(result.stdout)
        sys.stderr.write(result.stderr)
        sys.exit(result.returncode)
    for line in result.stderr.splitlines():
        if line and not any(pattern.search(line) for pattern in CONFLICT_PATTERNS):
            print(line)

pip_install_quiet('.')
pip_install_quiet('protobuf==5.29.4', 'huggingface_hub==0.36.0', 'kagglehub==0.3.13')
subprocess.run([sys.executable, 'tools/kaggle_bootstrap.py', '--project-root', '/kaggle/working/HRVC'], check=True)
print('[RVC] 依赖与标准 v2 48k 底模校验完成')

In [ ]:
import hashlib, hmac, os, stat, subprocess, urllib.request
binary = '/kaggle/working/HRVC/cloudflared-linux-amd64'
cloudflared_url = 'https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64'
cloudflared_sha256 = '9d71c677db00134c1bd4144b7783486b654ad281b1ea62b4972098d19f770f17'
if not os.path.isfile(binary) or not hmac.compare_digest(hashlib.sha256(open(binary, 'rb').read()).hexdigest(), cloudflared_sha256):
    urllib.request.urlretrieve(cloudflared_url, binary)
actual = hashlib.sha256(open(binary, 'rb').read()).hexdigest()
assert hmac.compare_digest(actual, cloudflared_sha256), f'cloudflared SHA-256 校验失败: {actual}'
os.chmod(binary, os.stat(binary).st_mode | stat.S_IXUSR)
subprocess.run(['python', '-u', 'tools/kaggle_launch.py'], check=True)